In [51]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from scipy.stats import norm

parquet_df = pd.read_parquet("../data/processed/iv_surface.parquet")

## Baseline 1: Polynomial Regression

Features: logMoneyness, years to expiry.

### Preprocessing

Target: log(normalizedIV). Features standardized before fitting; Ridge regularisation. See report Data/Methodology sections for rationale.

In [52]:
train_df = parquet_df[parquet_df["split"] == "train"]
val_df = parquet_df[parquet_df["split"] == "val"]

poly_train_data = train_df[["logMoneyness", "years"]]
poly_val_data = val_df[["logMoneyness", "years"]]
iv_train_data = train_df[["normalizedIV"]]
iv_val_data = val_df[["normalizedIV"]]

## Normalized IV is positive and right-skewed, so we do log transform to make it more stable
iv_train_data = np.log(iv_train_data)
iv_val_data = np.log(iv_val_data)

## Standardize Features
scaler = StandardScaler()
poly_train_scaled = scaler.fit_transform(
    poly_train_data
)
poly_val_scaled = scaler.transform(poly_val_data)

Black-Scholes helpers and butterfly/calendar arbitrage checks (reused by `svi.ipynb` and `nn_utils.py`).

In [53]:
def black_scholes_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def black_scholes_put(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def compute_pred_prices(df, pred_iv):
    S, K, T, r = df["uClose"].values, df["okey_xx"].values, df["years"].values, df["rate"].values
    is_call = df["callPut"].values == 1
    return np.where(
        is_call,
        black_scholes_call(S, K, T, r, pred_iv),
        black_scholes_put(S, K, T, r, pred_iv)
    )

def butterfly_violations(df, tol=1e-8):
    """
    Vectorized butterfly arbitrage check.
    Within each (ticker, date, maturity, callPut) group, sorted by strike,
    the second derivative of option price w.r.t. strike must be >= 0.
    """
    grp = ["ticker", "tradingDate", "years", "callPut"]
    df = df.sort_values(grp + ["okey_xx"])

    p_prev = df.groupby(grp)["pred_price"].shift(1)
    p_next = df.groupby(grp)["pred_price"].shift(-1)
    k_prev = df.groupby(grp)["okey_xx"].shift(1)
    k_next = df.groupby(grp)["okey_xx"].shift(-1)

    h1 = df["okey_xx"] - k_prev
    h2 = k_next - df["okey_xx"]
    valid = (h1 > 0) & (h2 > 0)

    second_deriv = (2.0 / (h1 + h2)) * (
        (p_next - df["pred_price"]) / h2 - (df["pred_price"] - p_prev) / h1
    )

    checks     = int(valid.sum())
    violations = int((valid & (second_deriv < -tol)).sum())
    return violations / checks if checks > 0 else 0.0, checks

def calendar_violations(df, tol=1e-8):
    """
    Vectorized calendar spread arbitrage check.
    Within each (ticker, date, strike, callPut) group, sorted by maturity,
    total implied variance must be non-decreasing in time to expiry.
    """
    grp = ["ticker", "tradingDate", "okey_xx", "callPut"]
    df = df.sort_values(grp + ["years"])

    tv_prev = df.groupby(grp)["pred_total_var"].shift(1)
    valid = tv_prev.notna()

    checks     = int(valid.sum())
    violations = int((valid & (tv_prev > df["pred_total_var"] + tol)).sum())
    return violations / checks if checks > 0 else 0.0, checks

### Polynomial Degree Sweep (alpha=1.0)

Degrees 1, 2, 5, 8, 10 at fixed alpha. See report Methodology for degree-selection rationale.

In [54]:
degrees = [1, 2, 5, 8, 10]
sweep_results = []

for degree in degrees:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_poly = poly.fit_transform(poly_train_scaled)
    X_val_poly   = poly.transform(poly_val_scaled)

    poly_scaler = StandardScaler()
    X_train_poly = poly_scaler.fit_transform(X_train_poly)
    X_val_poly   = poly_scaler.transform(X_val_poly)

    model = Ridge(alpha=1.0)
    model.fit(X_train_poly, iv_train_data)

    train_pred   = np.exp(model.predict(X_train_poly)).ravel()
    val_pred     = np.exp(model.predict(X_val_poly)).ravel()
    train_actual = np.exp(iv_train_data.values).ravel()
    val_actual   = np.exp(iv_val_data.values).ravel()

    train_rmse_vol = np.sqrt(mean_squared_error(
        train_actual * train_df["atmIV"].values,
        train_pred   * train_df["atmIV"].values
    ))
    val_rmse_vol = np.sqrt(mean_squared_error(
        val_actual * val_df["atmIV"].values,
        val_pred   * val_df["atmIV"].values
    ))

    # Violation checks on val only - train violations don't inform degree selection
    val_pred_midIV = val_pred * val_df["atmIV"].values
    val_df_copy = val_df.copy()
    val_df_copy["pred_price"]     = compute_pred_prices(val_df, val_pred_midIV)
    val_df_copy["pred_total_var"] = val_pred_midIV ** 2 * val_df["years"].values

    val_bf_rate,  val_bf_count  = butterfly_violations(val_df_copy)
    val_cal_rate, val_cal_count = calendar_violations(val_df_copy)

    sweep_results.append({
        "degree": degree, "split": "train",
        "rmse_volpts": train_rmse_vol * 100,
        "bf_violation_pct": None, "cal_violation_pct": None,
    })
    sweep_results.append({
        "degree": degree, "split": "val",
        "rmse_volpts": val_rmse_vol * 100,
        "bf_violation_pct": val_bf_rate * 100,
        "cal_violation_pct": val_cal_rate * 100,
    })

print(f"Val butterfly checks: {val_bf_count:,}  |  Val calendar checks: {val_cal_count:,}\n")
sweep_df = pd.DataFrame(sweep_results).set_index(["degree", "split"])
sweep_df.round({"rmse_volpts": 2, "bf_violation_pct": 4, "cal_violation_pct": 4})

Val butterfly checks: 772,371  |  Val calendar checks: 699,702



rmse_volpts  bf_violation_pct  cal_violation_pct
degree split                                                  
1      train        13.84               NaN                NaN
       val          11.87            0.0799             0.0000
2      train        13.23               NaN                NaN
       val          11.08            0.0012             0.4450
5      train        13.83               NaN                NaN
       val          11.22            0.2178             0.5385
8      train        13.66               NaN                NaN
       val          10.87            0.4728             0.8964
10     train        13.60               NaN                NaN
       val          10.85            0.5250             0.5654

### Alpha Tuning — Degree 5

40 log-spaced alpha values (0.1 to 1e8), chronological validation set. See report Methodology for the RMSE/arbitrage trade-off discussion.

In [55]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DEGREE = 5
alphas = np.logspace(-1, 8, 40)

poly = PolynomialFeatures(degree=DEGREE, include_bias=False)
X_train_poly = poly.fit_transform(poly_train_scaled)
X_val_poly   = poly.transform(poly_val_scaled)

poly_scaler = StandardScaler()
X_train_poly = poly_scaler.fit_transform(X_train_poly)
X_val_poly   = poly_scaler.transform(X_val_poly)

results = []
for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train_poly, iv_train_data)

    train_pred   = np.exp(model.predict(X_train_poly)).ravel()
    val_pred     = np.exp(model.predict(X_val_poly)).ravel()
    train_actual = np.exp(iv_train_data.values).ravel()
    val_actual   = np.exp(iv_val_data.values).ravel()

    train_rmse_vol = np.sqrt(mean_squared_error(
        train_actual * train_df["atmIV"].values,
        train_pred   * train_df["atmIV"].values
    ))
    val_rmse_vol = np.sqrt(mean_squared_error(
        val_actual * val_df["atmIV"].values,
        val_pred   * val_df["atmIV"].values
    ))

    val_pred_midIV = val_pred * val_df["atmIV"].values
    val_df_copy = val_df.copy()
    val_df_copy["pred_price"]     = compute_pred_prices(val_df, val_pred_midIV)
    val_df_copy["pred_total_var"] = val_pred_midIV ** 2 * val_df["years"].values

    val_bf_rate,  _ = butterfly_violations(val_df_copy)
    val_cal_rate, _ = calendar_violations(val_df_copy)

    results.append({
        "alpha": alpha,
        "train_rmse_vol": train_rmse_vol,
        "val_rmse_vol": val_rmse_vol,
        "val_bf_rate": val_bf_rate,
        "val_cal_rate": val_cal_rate,
    })
    print(f"alpha={alpha:.2e} | train={train_rmse_vol*100:.2f} vol pts | val={val_rmse_vol*100:.2f} vol pts | bf={val_bf_rate:.4%} | cal={val_cal_rate:.4%}")

import pandas as pd
results_df = pd.DataFrame(results)
best = results_df.loc[results_df["val_rmse_vol"].idxmin()]
print(f"\nBest alpha: {best['alpha']:.4e}")
print(f"  Val RMSE:              {best['val_rmse_vol']*100:.2f} vol pts")
print(f"  Val butterfly rate:    {best['val_bf_rate']:.4%}")
print(f"  Val calendar rate:     {best['val_cal_rate']:.4%}")

alpha=1.00e-01 | train=13.83 vol pts | val=11.22 vol pts | bf=0.2179% | cal=0.5391%
alpha=1.70e-01 | train=13.83 vol pts | val=11.22 vol pts | bf=0.2179% | cal=0.5391%
alpha=2.89e-01 | train=13.83 vol pts | val=11.22 vol pts | bf=0.2179% | cal=0.5391%
alpha=4.92e-01 | train=13.83 vol pts | val=11.22 vol pts | bf=0.2178% | cal=0.5387%
alpha=8.38e-01 | train=13.83 vol pts | val=11.22 vol pts | bf=0.2178% | cal=0.5385%
alpha=1.43e+00 | train=13.83 vol pts | val=11.22 vol pts | bf=0.2178% | cal=0.5381%
alpha=2.42e+00 | train=13.83 vol pts | val=11.22 vol pts | bf=0.2179% | cal=0.5379%
alpha=4.12e+00 | train=13.83 vol pts | val=11.22 vol pts | bf=0.2173% | cal=0.5371%
alpha=7.02e+00 | train=13.83 vol pts | val=11.22 vol pts | bf=0.2163% | cal=0.5368%
alpha=1.19e+01 | train=13.82 vol pts | val=11.21 vol pts | bf=0.2153% | cal=0.5347%
alpha=2.03e+01 | train=13.82 vol pts | val=11.21 vol pts | bf=0.2135% | cal=0.5324%
alpha=3.46e+01 | train=13.81 vol pts | val=11.20 vol pts | bf=0.2114% | cal=

Best alpha selected on validation RMSE. See report Methodology for the butterfly/calendar trade-off discussion.

In [ ]:
from pathlib import Path

results_dir = Path("../results")
figures_dir = results_dir / "figures" / "polynomial"
figures_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(results_dir / "poly_alpha_search.csv", index=False)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

ax1.semilogx(results_df["alpha"], results_df["val_rmse_vol"],   "b-o",  markersize=3, label="Val RMSE")
ax1.semilogx(results_df["alpha"], results_df["train_rmse_vol"], "b--",  markersize=3, alpha=0.5, label="Train RMSE")
ax2.semilogx(results_df["alpha"], results_df["val_bf_rate"]  * 100, "r-s", markersize=3, label="Val BF violation %")
ax2.semilogx(results_df["alpha"], results_df["val_cal_rate"] * 100, "r--", markersize=3, alpha=0.6, label="Val Cal violation %")

ax1.axvline(best["alpha"], color="gray", linestyle=":", label=f"Best alpha={best['alpha']:.2e}")
ax1.set_xlabel("Alpha (log scale)")
ax1.set_ylabel("RMSE (vol points)", color="b")
ax2.set_ylabel("Violation rate (%)", color="r")
ax1.set_title("Polynomial Regression (degree 5) - Alpha Tuning")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)

plt.tight_layout()
plt.savefig(figures_dir / "poly_alpha_search.png", dpi=150)
plt.show()
print(f"Saved to {results_dir / 'poly_alpha_search.csv'} and {figures_dir / 'poly_alpha_search.png'}")

### Final Model — Test Set Evaluation

Refit on the full training set at the selected (degree, alpha); evaluated once on each test split. See report Data section for the `midIV` sentinel-filter note.

In [57]:
BEST_DEGREE = 5
BEST_ALPHA  = best["alpha"]

# Load test sets
test_temp_df   = parquet_df[parquet_df["split"] == "test_temporal"]
test_ticker_df = parquet_df[parquet_df["split"] == "test_ticker"]

# Transform features using the same scaler fit on train
poly_test_temp_scaled   = scaler.transform(test_temp_df[["logMoneyness", "years"]])
poly_test_ticker_scaled = scaler.transform(test_ticker_df[["logMoneyness", "years"]])

# Build polynomial features - fit on train, transform all sets
poly_final = PolynomialFeatures(degree=BEST_DEGREE, include_bias=False)
X_train_final = poly_final.fit_transform(poly_train_scaled)
X_test_temp   = poly_final.transform(poly_test_temp_scaled)
X_test_ticker = poly_final.transform(poly_test_ticker_scaled)

poly_scaler_final = StandardScaler()
X_train_final = poly_scaler_final.fit_transform(X_train_final)
X_test_temp   = poly_scaler_final.transform(X_test_temp)
X_test_ticker = poly_scaler_final.transform(X_test_ticker)

# Fit final model on full training set
final_model = Ridge(alpha=BEST_ALPHA)
final_model.fit(X_train_final, iv_train_data)

def evaluate_split(df, X, label):
    actual_normIV = df["normalizedIV"].values
    pred_normIV   = np.exp(final_model.predict(X)).ravel()
    pred_midIV    = pred_normIV * df["atmIV"].values

    rmse_normIV = np.sqrt(mean_squared_error(actual_normIV, pred_normIV))
    rmse_vol    = np.sqrt(mean_squared_error(
        actual_normIV * df["atmIV"].values,
        pred_midIV
    ))

    # Out-of-sample R2 - listed first in the proposal's evaluation plan (section 5).
    # Reported in normalizedIV space so it is directly comparable across models.
    # Note both the error and the target variance are affected by the midIV
    # sentinel filter, so R2 and RMSE do not move proportionally between splits.
    r2 = 1 - mean_squared_error(actual_normIV, pred_normIV) / np.var(actual_normIV)

    # Butterfly violations
    df_copy = df.copy()
    df_copy["pred_price"]     = compute_pred_prices(df, pred_midIV)
    bf_rate, bf_checks = butterfly_violations(df_copy)

    # Calendar violations - total variance must be non-decreasing in maturity at fixed strike
    df_copy["pred_total_var"] = pred_midIV ** 2 * df["years"].values
    cal_rate, cal_checks = calendar_violations(df_copy)

    print(f"{label}")
    print(f"  R2          = {r2:.4f}")
    print(f"  RMSE        = {rmse_normIV:.4f} (normalizedIV) = {rmse_vol*100:.2f} vol pts")
    print(f"  Butterfly   = {bf_rate:.4%} of {bf_checks:,} checks")
    print(f"  Calendar    = {cal_rate:.4%} of {cal_checks:,} checks")
    print()

    return {
        "split": label,
        "r2": r2,
        "rmse_normIV": rmse_normIV,
        "rmse_volpts": rmse_vol * 100,
        "bf_violation_pct": bf_rate * 100,
        "bf_checks": bf_checks,
        "cal_violation_pct": cal_rate * 100,
        "cal_checks": cal_checks,
    }

print(f"Final model: degree={BEST_DEGREE}, alpha={BEST_ALPHA:.2e}\n")
test_results = []
test_results.append(evaluate_split(test_temp_df,   X_test_temp,   "test_temporal  (trained tickers, Nov–Dec)"))
test_results.append(evaluate_split(test_ticker_df, X_test_ticker, "test_ticker    (held-out tickers, Nov–Dec)"))

test_results_df = pd.DataFrame(test_results).set_index("split")
test_results_df.to_csv(results_dir / "poly_test_results.csv")
print(test_results_df.round({"r2": 4, "rmse_normIV": 4, "rmse_volpts": 2, "bf_violation_pct": 4, "cal_violation_pct": 4}))
print(f"\nSaved to {results_dir / 'poly_test_results.csv'}")

Final model: degree=5, alpha=2.89e+05

test_temporal  (trained tickers, Nov–Dec)
  R2          = 0.1610
  RMSE        = 0.6890 (normalizedIV) = 11.74 vol pts
  Butterfly   = 0.0000% of 694,082 checks
  Calendar    = 0.3080% of 627,521 checks

test_ticker    (held-out tickers, Nov–Dec)
  R2          = 0.2288
  RMSE        = 0.3283 (normalizedIV) = 15.03 vol pts
  Butterfly   = 0.0257% of 264,586 checks
  Calendar    = 0.4557% of 245,981 checks

                                                r2  rmse_normIV  rmse_volpts  \
split                                                                          
test_temporal  (trained tickers, Nov–Dec)   0.1610       0.6890        11.74   
test_ticker    (held-out tickers, Nov–Dec)  0.2288       0.3283        15.03   

                                            bf_violation_pct  bf_checks  \
split                                                                     
test_temporal  (trained tickers, Nov–Dec)             0.0000     694082   
test_t

---
### Takeaways

See report Results and Methodology for interpretation of these numbers.

# 5-Feature Polynomial

Adds atmIV, realizedVol, recentReturn to logMoneyness and years.

In [58]:
poly_train_data = train_df[["logMoneyness", "years", "atmIV", "realizedVol", "recentReturn"]]
poly_val_data = val_df[["logMoneyness", "years", "atmIV", "realizedVol", "recentReturn"]]

## Standardize Features
scaler = StandardScaler()
poly_train_scaled = scaler.fit_transform(
    poly_train_data
)
poly_val_scaled = scaler.transform(poly_val_data)

In [59]:
# Since we already have 5 features, going above this degree is likely to overfit and will take a very long time to run
degrees = [1, 2, 3]
sweep_results = []

for degree in degrees:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_poly = poly.fit_transform(poly_train_scaled)
    X_val_poly   = poly.transform(poly_val_scaled)

    poly_scaler = StandardScaler()
    X_train_poly = poly_scaler.fit_transform(X_train_poly)
    X_val_poly   = poly_scaler.transform(X_val_poly)

    model = Ridge(alpha=1.0)
    model.fit(X_train_poly, iv_train_data)

    train_pred   = np.exp(model.predict(X_train_poly)).ravel()
    val_pred     = np.exp(model.predict(X_val_poly)).ravel()
    train_actual = np.exp(iv_train_data.values).ravel()
    val_actual   = np.exp(iv_val_data.values).ravel()

    train_rmse_vol = np.sqrt(mean_squared_error(
        train_actual * train_df["atmIV"].values,
        train_pred   * train_df["atmIV"].values
    ))
    val_rmse_vol = np.sqrt(mean_squared_error(
        val_actual * val_df["atmIV"].values,
        val_pred   * val_df["atmIV"].values
    ))

    # Violation checks on val only - train violations don't inform degree selection
    val_pred_midIV = val_pred * val_df["atmIV"].values
    val_df_copy = val_df.copy()
    val_df_copy["pred_price"]     = compute_pred_prices(val_df, val_pred_midIV)
    val_df_copy["pred_total_var"] = val_pred_midIV ** 2 * val_df["years"].values

    val_bf_rate,  val_bf_count  = butterfly_violations(val_df_copy)
    val_cal_rate, val_cal_count = calendar_violations(val_df_copy)

    sweep_results.append({
        "degree": degree, "split": "train",
        "rmse_volpts": train_rmse_vol * 100,
        "bf_violation_pct": None, "cal_violation_pct": None,
    })
    sweep_results.append({
        "degree": degree, "split": "val",
        "rmse_volpts": val_rmse_vol * 100,
        "bf_violation_pct": val_bf_rate * 100,
        "cal_violation_pct": val_cal_rate * 100,
    })

print(f"Val butterfly checks: {val_bf_count:,}  |  Val calendar checks: {val_cal_count:,}\n")
sweep_df = pd.DataFrame(sweep_results).set_index(["degree", "split"])
sweep_df.round({"rmse_volpts": 2, "bf_violation_pct": 4, "cal_violation_pct": 4})

Val butterfly checks: 772,371  |  Val calendar checks: 699,702



rmse_volpts  bf_violation_pct  cal_violation_pct
degree split                                                  
1      train        10.48               NaN                NaN
       val          10.39            0.0513             0.0000
2      train        10.82               NaN                NaN
       val           9.48            0.0149             0.6168
3      train         8.35               NaN                NaN
       val           8.45            0.0306             0.7935

### Alpha Tuning — Degree 3

Degree swept 1–3 only (see report Methodology). Alpha tuned as above.

In [60]:
DEGREE_5FEAT = 3
alphas_5feat = np.logspace(-1, 8, 40)

poly_5feat = PolynomialFeatures(degree=DEGREE_5FEAT, include_bias=False)
X_train_poly_5 = poly_5feat.fit_transform(poly_train_scaled)
X_val_poly_5   = poly_5feat.transform(poly_val_scaled)

poly_scaler_5 = StandardScaler()
X_train_poly_5 = poly_scaler_5.fit_transform(X_train_poly_5)
X_val_poly_5   = poly_scaler_5.transform(X_val_poly_5)

results_5feat = []
for alpha in alphas_5feat:
    model = Ridge(alpha=alpha)
    model.fit(X_train_poly_5, iv_train_data)

    train_pred   = np.exp(model.predict(X_train_poly_5)).ravel()
    val_pred     = np.exp(model.predict(X_val_poly_5)).ravel()
    train_actual = np.exp(iv_train_data.values).ravel()
    val_actual   = np.exp(iv_val_data.values).ravel()

    train_rmse_vol = np.sqrt(mean_squared_error(
        train_actual * train_df["atmIV"].values,
        train_pred   * train_df["atmIV"].values
    ))
    val_rmse_vol = np.sqrt(mean_squared_error(
        val_actual * val_df["atmIV"].values,
        val_pred   * val_df["atmIV"].values
    ))

    val_pred_midIV = val_pred * val_df["atmIV"].values
    val_df_copy = val_df.copy()
    val_df_copy["pred_price"]     = compute_pred_prices(val_df, val_pred_midIV)
    val_df_copy["pred_total_var"] = val_pred_midIV ** 2 * val_df["years"].values

    val_bf_rate,  _ = butterfly_violations(val_df_copy)
    val_cal_rate, _ = calendar_violations(val_df_copy)

    results_5feat.append({
        "alpha": alpha,
        "train_rmse_vol": train_rmse_vol,
        "val_rmse_vol": val_rmse_vol,
        "val_bf_rate": val_bf_rate,
        "val_cal_rate": val_cal_rate,
    })
    print(f"alpha={alpha:.2e} | train={train_rmse_vol*100:.2f} vol pts | val={val_rmse_vol*100:.2f} vol pts | bf={val_bf_rate:.4%} | cal={val_cal_rate:.4%}")

results_df_5feat = pd.DataFrame(results_5feat)
best_5feat = results_df_5feat.loc[results_df_5feat["val_rmse_vol"].idxmin()]
print(f"\nBest alpha: {best_5feat['alpha']:.4e}")
print(f"  Val RMSE:              {best_5feat['val_rmse_vol']*100:.2f} vol pts")
print(f"  Val butterfly rate:    {best_5feat['val_bf_rate']:.4%}")
print(f"  Val calendar rate:     {best_5feat['val_cal_rate']:.4%}")

alpha=1.00e-01 | train=8.35 vol pts | val=8.45 vol pts | bf=0.0306% | cal=0.7935%
alpha=1.70e-01 | train=8.35 vol pts | val=8.45 vol pts | bf=0.0306% | cal=0.7935%
alpha=2.89e-01 | train=8.35 vol pts | val=8.45 vol pts | bf=0.0306% | cal=0.7935%
alpha=4.92e-01 | train=8.35 vol pts | val=8.45 vol pts | bf=0.0306% | cal=0.7935%
alpha=8.38e-01 | train=8.35 vol pts | val=8.45 vol pts | bf=0.0306% | cal=0.7935%
alpha=1.43e+00 | train=8.35 vol pts | val=8.45 vol pts | bf=0.0306% | cal=0.7935%
alpha=2.42e+00 | train=8.35 vol pts | val=8.45 vol pts | bf=0.0306% | cal=0.7935%
alpha=4.12e+00 | train=8.35 vol pts | val=8.45 vol pts | bf=0.0306% | cal=0.7935%
alpha=7.02e+00 | train=8.35 vol pts | val=8.45 vol pts | bf=0.0306% | cal=0.7935%
alpha=1.19e+01 | train=8.35 vol pts | val=8.45 vol pts | bf=0.0306% | cal=0.7932%
alpha=2.03e+01 | train=8.35 vol pts | val=8.45 vol pts | bf=0.0307% | cal=0.7932%
alpha=3.46e+01 | train=8.35 vol pts | val=8.45 vol pts | bf=0.0308% | cal=0.7929%
alpha=5.88e+01 |

Best alpha selected on validation RMSE.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

ax1.semilogx(results_df_5feat["alpha"], results_df_5feat["val_rmse_vol"],   "b-o",  markersize=3, label="Val RMSE")
ax1.semilogx(results_df_5feat["alpha"], results_df_5feat["train_rmse_vol"], "b--",  markersize=3, alpha=0.5, label="Train RMSE")
ax2.semilogx(results_df_5feat["alpha"], results_df_5feat["val_bf_rate"]  * 100, "r-s", markersize=3, label="Val BF violation %")
ax2.semilogx(results_df_5feat["alpha"], results_df_5feat["val_cal_rate"] * 100, "r--", markersize=3, alpha=0.6, label="Val Cal violation %")

ax1.axvline(best_5feat["alpha"], color="gray", linestyle=":", label=f"Best alpha={best_5feat['alpha']:.2e}")
ax1.set_xlabel("Alpha (log scale)")
ax1.set_ylabel("RMSE (vol points)", color="b")
ax2.set_ylabel("Violation rate (%)", color="r")
ax1.set_title("Polynomial Regression (5 features, degree 3) - Alpha Tuning")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)

plt.tight_layout()
plt.savefig(figures_dir / "poly_alpha_search_5feat.png", dpi=150)
plt.show()
print(f"Saved to {figures_dir / 'poly_alpha_search_5feat.png'}")

### Final Model — Test Set Evaluation (5 Features)

Features: logMoneyness, years, atmIV, realizedVol, recentReturn.

In [62]:
BEST_DEGREE_5FEAT = DEGREE_5FEAT
BEST_ALPHA_5FEAT  = best_5feat["alpha"]

FEATURES_5 = ["logMoneyness", "years", "atmIV", "realizedVol", "recentReturn"]

poly_test_temp_scaled_5   = scaler.transform(test_temp_df[FEATURES_5])
poly_test_ticker_scaled_5 = scaler.transform(test_ticker_df[FEATURES_5])

poly_final_5 = PolynomialFeatures(degree=BEST_DEGREE_5FEAT, include_bias=False)
X_train_final_5 = poly_final_5.fit_transform(poly_train_scaled)
X_test_temp_5   = poly_final_5.transform(poly_test_temp_scaled_5)
X_test_ticker_5 = poly_final_5.transform(poly_test_ticker_scaled_5)

poly_scaler_final_5 = StandardScaler()
X_train_final_5 = poly_scaler_final_5.fit_transform(X_train_final_5)
X_test_temp_5   = poly_scaler_final_5.transform(X_test_temp_5)
X_test_ticker_5 = poly_scaler_final_5.transform(X_test_ticker_5)

final_model_5feat = Ridge(alpha=BEST_ALPHA_5FEAT)
final_model_5feat.fit(X_train_final_5, iv_train_data)

def evaluate_split_5feat(df, X, label):
    actual_normIV = df["normalizedIV"].values
    pred_normIV   = np.exp(final_model_5feat.predict(X)).ravel()
    pred_midIV    = pred_normIV * df["atmIV"].values

    rmse_normIV = np.sqrt(mean_squared_error(actual_normIV, pred_normIV))
    rmse_vol    = np.sqrt(mean_squared_error(
        actual_normIV * df["atmIV"].values,
        pred_midIV
    ))
    r2 = 1 - mean_squared_error(actual_normIV, pred_normIV) / np.var(actual_normIV)

    df_copy = df.copy()
    df_copy["pred_price"] = compute_pred_prices(df, pred_midIV)
    bf_rate, bf_checks = butterfly_violations(df_copy)

    df_copy["pred_total_var"] = pred_midIV ** 2 * df["years"].values
    cal_rate, cal_checks = calendar_violations(df_copy)

    print(f"{label}")
    print(f"  R2          = {r2:.4f}")
    print(f"  RMSE        = {rmse_normIV:.4f} (normalizedIV) = {rmse_vol*100:.2f} vol pts")
    print(f"  Butterfly   = {bf_rate:.4%} of {bf_checks:,} checks")
    print(f"  Calendar    = {cal_rate:.4%} of {cal_checks:,} checks")
    print()

    return {
        "split": label,
        "r2": r2,
        "rmse_normIV": rmse_normIV,
        "rmse_volpts": rmse_vol * 100,
        "bf_violation_pct": bf_rate * 100,
        "bf_checks": bf_checks,
        "cal_violation_pct": cal_rate * 100,
        "cal_checks": cal_checks,
    }

print(f"Final model (5 features): degree={BEST_DEGREE_5FEAT}, alpha={BEST_ALPHA_5FEAT:.2e}\n")
test_results_5feat = []
test_results_5feat.append(evaluate_split_5feat(test_temp_df,   X_test_temp_5,   "test_temporal  (trained tickers, Nov–Dec)"))
test_results_5feat.append(evaluate_split_5feat(test_ticker_df, X_test_ticker_5, "test_ticker    (held-out tickers, Nov–Dec)"))

test_results_df_5feat = pd.DataFrame(test_results_5feat).set_index("split")
print(test_results_df_5feat.round({"r2": 4, "rmse_normIV": 4, "rmse_volpts": 2, "bf_violation_pct": 4, "cal_violation_pct": 4}))

Final model (5 features): degree=3, alpha=1.00e+05

test_temporal  (trained tickers, Nov–Dec)
  R2          = 0.5074
  RMSE        = 0.5280 (normalizedIV) = 10.30 vol pts
  Butterfly   = 0.0413% of 694,082 checks
  Calendar    = 0.8535% of 627,521 checks

test_ticker    (held-out tickers, Nov–Dec)
  R2          = -0.0443
  RMSE        = 0.3821 (normalizedIV) = 20.57 vol pts
  Butterfly   = 0.3058% of 264,586 checks
  Calendar    = 0.4574% of 245,981 checks

                                                r2  rmse_normIV  rmse_volpts  \
split                                                                          
test_temporal  (trained tickers, Nov–Dec)   0.5074       0.5280        10.30   
test_ticker    (held-out tickers, Nov–Dec) -0.0443       0.3821        20.57   

                                            bf_violation_pct  bf_checks  \
split                                                                     
test_temporal  (trained tickers, Nov–Dec)             0.0413     69

In [63]:
test_results_df_5feat.to_csv(results_dir / "poly_test_results_5feat.csv")
print(f"Saved to {results_dir / 'poly_test_results_5feat.csv'}")

Saved to ../results/poly_test_results_5feat.csv


---
### Takeaways

See report Results for interpretation of these numbers relative to the 2-feature baseline and the neural network.